# Training — Variational Autoencoder (VAE)

<div style="text-align: justify">

The following notebook trains a <b>Variational Autoencoder (VAE)</b> for the <b>Tau Anomaly Detection</b> analysis. The model is trained on <b>background-only</b> events using PyTorch Lightning. Signal events are held out and used only to visualise reconstruction error separation after training. In addition to reconstruction diagnostics, this notebook includes <b>VAE-specific latent space analysis</b> and <b>KL divergence monitoring</b> for posterior collapse detection. Experiment metrics are tracked with <b>Weights & Biases</b>.

</div>

## Pipeline Summary

| Step | Module | Description |
|------|--------|-------------|
| Config | `hydra.compose` | Load analysis and model configuration |
| DataModule | `datamodule.AnomalyDataModule` | Read mc.parquet, split bkg/sig, fit scaler, build dataloaders |
| Model | `vae.VariationalAutoencoder` | Instantiate VAE with reparameterization trick |
| Logger | `WandbLogger` | Initialise Weights & Biases experiment tracking |
| Train | `lightning.Trainer` | Fit model with early stopping and checkpointing |
| Scores | `anomaly.reconstruction_error` | Compute per-event anomaly scores on predict set |
| Diagnostics | `plots` | Loss components, reconstruction error, latent space analysis |

The same pipeline is available as a CLI via `uv run python run.py stage=train model=vae`.

## Initialization

### Libraries

Configuration:
* [Hydra](https://hydra.cc/)
* [OmegaConf](https://omegaconf.readthedocs.io/)
* [pyrootutils](https://github.com/ashleve/pyrootutils)

Data Processing:
* [Pandas](https://pandas.pydata.org/)

Machine Learning:
* [PyTorch](https://pytorch.org/)
* [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/)

Experiment Tracking:
* [Weights & Biases](https://wandb.ai/)

### Notebook

Activating autoreload of imported modules.

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

Initializing the project root.

In [ ]:
import pyrootutils

path = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

Suppressing unessential warnings and applying ATLAS style.

In [ ]:
from src.utils import suppress_warnings
from src.visualization.plots import apply_atlas_style

suppress_warnings()
apply_atlas_style()

## Configuration

Loading the Hydra configuration with the VAE model config.

In [ ]:
from hydra import compose, initialize_config_dir

initialize_config_dir(config_dir=str(path / "configs"), version_base="1.3")
cfg = compose(config_name="config", overrides=["model=vae"])

Resolving input and output directories from config.

In [ ]:
from src.processing.analysis import get_output_paths

output_paths = get_output_paths(cfg)
dataframes_dir = path / output_paths["dataframes_dir"]
models_dir = path / output_paths["models_dir"]
plots_dir = path / output_paths["plots_dir"] / "vae"

models_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

## DataModule

Setting up the `AnomalyDataModule`. It reads the MC parquet, separates background from signal, fits a scaler on the background training split, and builds `TensorDataset` objects that pair features with event weights.

In [ ]:
import lightning as L

L.seed_everything(cfg.seed, workers=True)

In [ ]:
from src.models.datamodule import AnomalyDataModule

background_origins = {
    s["id"]
    for s in cfg.samples.background.samples
    if s["id"] not in set(cfg.samples.background.get("exclude", []))
}
print(f"Background origins: {background_origins}")

dm = AnomalyDataModule(
    mc_path=str(dataframes_dir / "mc.parquet"),
    background_origins=background_origins,
    normalization=cfg.model.normalization,
    val_fraction=cfg.pipeline.val_fraction,
    batch_size=cfg.model.batch_size,
    seed=cfg.seed,
)
dm.setup()
print(f"Features: {dm.n_features}")
print(f"Feature names: {dm.feature_names_}")

## Model

Instantiating the Variational Autoencoder from the typed config. The encoder outputs `mu` and `logvar` for the latent distribution; the decoder reconstructs from samples drawn via the reparameterization trick.

In [ ]:
from omegaconf import OmegaConf

from src.models.config import VAEConfig
from src.models.vae import VariationalAutoencoder

model_params = dict(OmegaConf.to_container(cfg.model, resolve=True))
model_cfg = VAEConfig(**model_params)
model = VariationalAutoencoder(model_cfg, n_features=dm.n_features)

n_params = sum(p.numel() for p in model.parameters())
print(f"VAE: {n_params:,} parameters")
print(f"Beta: {model_cfg.beta}, schedule: {model_cfg.beta_schedule}")
print(model)

## Training

### WandB Logger

Initialising the Weights & Biases logger. Set `enabled: false` in `configs/pipeline/default.yaml` to disable tracking.

In [ ]:
from lightning.pytorch.loggers import WandbLogger

wandb_cfg = cfg.pipeline.wandb
if wandb_cfg.enabled:
    wandb_logger = WandbLogger(
        project=wandb_cfg.project,
        name=f"{cfg.experiment_name}-vae",
        log_model=wandb_cfg.log_model,
        config=dict(OmegaConf.to_container(cfg.model, resolve=True)),
    )
else:
    wandb_logger = False

print(f"WandB logging: {'enabled' if wandb_cfg.enabled else 'disabled'}")

### Trainer

Creating the Lightning Trainer with early stopping and model checkpointing.

In [ ]:
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from src.models.callbacks import EpochProgressBar, MetricTracker

tracker = MetricTracker()

callbacks = [
    EarlyStopping(
        monitor=cfg.pipeline.monitor_metric,
        mode=cfg.pipeline.monitor_mode,
        patience=cfg.pipeline.early_stopping_patience,
        verbose=False,
    ),
    ModelCheckpoint(
        dirpath=models_dir,
        filename="vae-best",
        monitor=cfg.pipeline.monitor_metric,
        mode=cfg.pipeline.monitor_mode,
        save_top_k=1,
        verbose=False,
    ),
    EpochProgressBar(),
    tracker,
]

trainer = L.Trainer(
    max_epochs=cfg.model.n_epochs,
    callbacks=callbacks,
    logger=wandb_logger,
    deterministic=True,
    precision="16-mixed" if cfg.model.amp else "32-true",
    enable_progress_bar=False,
)

### Fit

Training the VAE on background-only data. The VAE logs additional metrics: `recon_loss`, `kl_loss`, `beta`, `mu_mean`, `mu_var`, and `logvar_mean` for collapse monitoring.

In [ ]:
trainer.fit(model, datamodule=dm)

### Checkpoint

Saving the final checkpoint (includes scaler state from the DataModule).

In [ ]:
from pathlib import Path

best_path = getattr(trainer.checkpoint_callback, "best_model_path", None)
if best_path:
    best_path = Path(best_path).relative_to(path)
print(f"Best checkpoint: {best_path}")

ckpt_path = models_dir / "vae.ckpt"
trainer.save_checkpoint(ckpt_path)
print(f"Saved checkpoint: {ckpt_path.relative_to(path)}")

## Diagnostics

### Training Metrics

Final training metrics from the Trainer.

In [ ]:
from src.models.plots import plot_loss, plot_loss_components

fig = plot_loss(tracker.history["train_loss"], tracker.history["val_loss"], title="VAE Loss Plot")
fig.savefig(plots_dir / "vae_loss.png", dpi=150, bbox_inches="tight")

fig = plot_loss_components(
    tracker.history["train_recon_loss"],
    tracker.history["train_kl_loss"],
    beta_values=tracker.history.get("train_beta"),
    title="VAE Loss Components",
)
fig.savefig(plots_dir / "vae_loss_components.png", dpi=150, bbox_inches="tight")

### Anomaly Scores

Computing reconstruction error on the predict set (background validation + signal). Higher reconstruction error indicates a more anomalous event.

In [ ]:
import torch
import numpy as np

from src.models.anomaly import reconstruction_error, compute_threshold, build_scores_frame

predictions = trainer.predict(model, datamodule=dm)
x_hat = torch.cat(predictions)
x_orig = torch.cat([batch[0] for batch in dm.predict_dataloader()])

scores = reconstruction_error(x_orig, x_hat).numpy()
print(f"Scores: {len(scores):,} events")
print(f"  Background mean: {scores[dm.predict_labels == 0].mean():.6f}")
print(f"  Signal mean:     {scores[dm.predict_labels == 1].mean():.6f}")

### Threshold

In [ ]:
bkg_scores = scores[dm.predict_labels == 0]
sig_scores = scores[dm.predict_labels == 1]

threshold = compute_threshold(
    bkg_scores,
    strategy=cfg.pipeline.threshold_strategy,
    percentile=cfg.pipeline.threshold_percentile,
)
print(f"Threshold ({cfg.pipeline.threshold_strategy}): {threshold:.6f}")
print(f"  Background flagged: {(bkg_scores > threshold).sum():,} / {len(bkg_scores):,}")
print(f"  Signal flagged:     {(sig_scores > threshold).sum():,} / {len(sig_scores):,}")

### Reconstruction Error Distribution

Comparing the reconstruction error distributions of background and signal events.

In [ ]:
from src.models.plots import plot_reconstruction_error

fig = plot_reconstruction_error(bkg_scores, sig_scores, threshold=threshold, title="VAE Reconstruction Error")
fig.savefig(plots_dir / "vae_reconstruction_error.png", dpi=150, bbox_inches="tight")

### Single Event Reconstruction

Bar chart comparing original vs reconstructed feature values for a single event.

In [ ]:
from src.models.plots import plot_reconstruction_performance

fig = plot_reconstruction_performance(
    x_orig[0].numpy(),
    x_hat[0].numpy(),
    dm.feature_names_,
    event_idx=0,
    title="VAE Single Event Reconstruction",
)
fig.savefig(plots_dir / "vae_single_event.png", dpi=150, bbox_inches="tight")

### Feature Histograms

Per-feature distributions of original vs reconstructed values across the predict set.

In [ ]:
from src.models.plots import plot_feature_histograms

fig = plot_feature_histograms(
    x_orig.numpy(),
    x_hat.numpy(),
    dm.feature_names_,
    title="VAE Feature Distributions",
)
fig.savefig(plots_dir / "vae_feature_histograms.png", dpi=150, bbox_inches="tight")

## Latent Space Analysis

Encoding the predict set to extract `mu` and `logvar` for latent diagnostics.

In [ ]:
model.eval()
all_mu, all_logvar = [], []
with torch.no_grad():
    for batch in dm.predict_dataloader():
        x, _w = batch
        mu, logvar = model.encode(x)
        all_mu.append(mu)
        all_logvar.append(logvar)

mu_np = torch.cat(all_mu).numpy()
logvar_np = torch.cat(all_logvar).numpy()
print(f"Latent space: {mu_np.shape[0]:,} events, {mu_np.shape[1]} dimensions")

### Latent Mean Spread

Variance of `mu` per latent dimension. Dimensions with variance below 0.1 indicate potential posterior collapse.

In [ ]:
from src.models.plots import plot_latent_mean_spread

fig = plot_latent_mean_spread(mu_np)
fig.savefig(plots_dir / "vae_mu_spread.png", dpi=150, bbox_inches="tight")

### Log-Variance Spread

Mean `logvar` per latent dimension. Dimensions with mean logvar below -5 indicate potential collapse.

In [ ]:
from src.models.plots import plot_logvar_spread

fig = plot_logvar_spread(logvar_np)
fig.savefig(plots_dir / "vae_logvar_spread.png", dpi=150, bbox_inches="tight")

### Mu vs Logvar

Scatter plot of mean `mu` vs mean `logvar` per latent dimension.

In [ ]:
from src.models.plots import plot_mu_vs_logvar

fig = plot_mu_vs_logvar(mu_np, logvar_np)
fig.savefig(plots_dir / "vae_mu_vs_logvar.png", dpi=150, bbox_inches="tight")

### KL per Dimension

Mean KL divergence per latent dimension. Identifies which dimensions encode the most information.

In [ ]:
from src.models.plots import plot_kl_per_dimension

kl_per_dim = -0.5 * (1 + logvar_np - mu_np ** 2 - np.exp(logvar_np)).mean(axis=0)
fig = plot_kl_per_dimension(kl_per_dim)
fig.savefig(plots_dir / "vae_kl_per_dim.png", dpi=150, bbox_inches="tight")

### Latent Mean Histograms

Per-dimension histograms of the latent mean `mu`.

In [ ]:
from src.models.plots import plot_latent_mean_histograms

fig = plot_latent_mean_histograms(mu_np)
fig.savefig(plots_dir / "vae_mu_histograms.png", dpi=150, bbox_inches="tight")

### Log-Variance Histograms

Per-dimension histograms of `logvar`.

In [ ]:
from src.models.plots import plot_logvar_histograms

fig = plot_logvar_histograms(logvar_np)
fig.savefig(plots_dir / "vae_logvar_histograms.png", dpi=150, bbox_inches="tight")

### Latent Dimension Histograms

Sampled latent vectors coloured by background vs signal.

In [ ]:
from src.models.plots import plot_latent_histograms

fig = plot_latent_histograms(mu_np, labels=dm.predict_labels)
fig.savefig(plots_dir / "vae_latent_histograms.png", dpi=150, bbox_inches="tight")

## Scores DataFrame

Building and saving the tidy scores DataFrame for downstream evaluation.

In [ ]:
scores_df = build_scores_frame(scores, dm.predict_labels, dm.predict_origins)
scores_path = dataframes_dir / "vae_scores.parquet"
scores_df.to_parquet(scores_path)
print(f"Saved scores: {scores_path}")
scores_df.head()

### Finish WandB

Closing the WandB run.

In [ ]:
import wandb

if wandb.run is not None:
    wandb.finish()